# Live autotuning: ArchTune vs unpruned

Tunes a 4096×4096×4096 fp16 matmul with `triton.autotune` twice. With ArchTune, all 384 configs
are scored and only the top 8 are compiled and benchmarked. Without it, all 384 are. Both runs start
from an empty Triton cache, so every config they try is compiled from scratch.

In [ ]:
import os, sys, pathlib
os.environ.setdefault("CC", os.path.expanduser("~/bin/zcc"))   # Triton builds a small C helper at runtime
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
from archtune import gpu, model, run
ENV = gpu.capture()
print(ENV["gpu_name"], "| torch", ENV["torch_version"], "| triton", ENV["triton_version"])
A = gpu.live()
models = run.load_models(ROOT / "results", ENV["gpu_tag"])
KW = {"reg_model": models[0], "smem_model": models[1]} if models else {}
CONFIGS = model.matmul_space()
M = N = K = 4096

## ArchTune, top 8

In [ ]:
pruned = model.tune_live(CONFIGS, M, N, K, prune_by=model.archtune_prune_configs_by(A, 8, **KW))
print(f"tuning {pruned['tuning_s']:.1f} s | kernel {pruned['time_ms_median']:.3f} ms | correct={pruned['correct']}")
print("chosen config:", pruned["best_cfg"])

## Unpruned, all 384 (this one takes a while)

In [ ]:
full = model.tune_live(CONFIGS, M, N, K, prune_by=None)
print(f"tuning {full['tuning_s']:.1f} s ({full['tuning_s'] / 60:.1f} min) | kernel {full['time_ms_median']:.3f} ms | correct={full['correct']}")
print("chosen config:", full["best_cfg"])

In [ ]:
print(f"{full['tuning_s'] / pruned['tuning_s']:.0f}x less tuning time; "
      f"kernel {(pruned['time_ms_median'] / full['time_ms_median'] - 1) * 100:+.1f}% vs the unpruned choice")